In [1]:
# Mount & installs (run once)
from google.colab import drive
drive.mount('/content/drive')

!pip install -q torch torchvision torchaudio scikit-image matplotlib tqdm pillow requests

import os, torch, torch.nn as nn, torch.optim as optim
from torchvision import datasets, transforms, utils
from torch.utils.data import DataLoader
from tqdm import tqdm
import numpy as np
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")
BASE="/content/drive/MyDrive/image_compression_project"
os.makedirs(BASE, exist_ok=True)


Mounted at /content/drive
Device: cuda


In [4]:
class VAE(nn.Module):
    def __init__(self, latent_dim=512):
        super().__init__()
        self.enc_conv = nn.Sequential(
            nn.Conv2d(3,32,4,2,1), nn.ReLU(),
            nn.Conv2d(32,64,4,2,1), nn.ReLU(),
            nn.Conv2d(64,128,4,2,1), nn.ReLU(),
            nn.Flatten()
        )
        self.fc_mu = nn.Linear(128*16*16, latent_dim)
        self.fc_logvar = nn.Linear(128*16*16, latent_dim)
        self.fc_dec = nn.Linear(latent_dim, 128*16*16)
        self.dec = nn.Sequential(
            nn.Unflatten(1,(128,16,16)),
            nn.ConvTranspose2d(128,64,4,2,1), nn.ReLU(),
            nn.ConvTranspose2d(64,32,4,2,1), nn.ReLU(),
            nn.ConvTranspose2d(32,3,4,2,1), nn.Sigmoid()
        )
    def reparam(self, mu, logvar):
        std = (0.5*logvar).exp()
        eps = torch.randn_like(std)
        return mu + eps*std
    def forward(self,x):
        h = self.enc_conv(x)
        mu = self.fc_mu(h); logvar = self.fc_logvar(h)
        z = self.reparam(mu, logvar)
        xrec = self.dec(self.fc_dec(z))
        return xrec, mu, logvar

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
vae = VAE(latent_dim=512).to(device)
opt = optim.Adam(vae.parameters(), lr=1e-3)
mse = nn.MSELoss()


In [5]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor()
])

train_ds = datasets.CIFAR10(root='data', train=True, download=True, transform=transform)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=2)

print("✅ Train loader ready:", len(train_ds), "images")


100%|██████████| 170M/170M [00:04<00:00, 41.7MB/s]


✅ Train loader ready: 50000 images


In [6]:
epochs = 10
beta = 1e-4   # start small and increase in later runs
for epoch in range(epochs):
    total=0
    vae.train()
    for imgs,_ in tqdm(train_loader, desc=f"VAE Epoch {epoch+1}/{epochs}"):
        imgs = imgs.to(device)
        rec, mu, logvar = vae(imgs)
        recon_loss = mse(rec, imgs)
        kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / imgs.size(0)
        loss = recon_loss + beta * kl
        opt.zero_grad(); loss.backward(); opt.step()
        total += loss.item()
    print(f"Epoch {epoch+1} avg {total/len(train_loader):.6f}")
torch.save(vae.state_dict(), os.path.join(BASE,"models/vae_epoch10.pth"))
print("Saved VAE ->", os.path.join(BASE,"models/vae_epoch10.pth"))


VAE Epoch 1/10: 100%|██████████| 782/782 [00:46<00:00, 16.65it/s]


Epoch 1 avg 0.026783


VAE Epoch 2/10: 100%|██████████| 782/782 [00:45<00:00, 17.30it/s]


Epoch 2 avg 0.019083


VAE Epoch 3/10: 100%|██████████| 782/782 [00:45<00:00, 17.14it/s]


...

Epoch 8 avg 0.014914


VAE Epoch 9/10: 100%|██████████| 782/782 [00:46<00:00, 16.99it/s]


Epoch 9 avg 0.014832


VAE Epoch 10/10: 100%|██████████| 782/782 [00:46<00:00, 16.93it/s]


Epoch 10 avg 0.014777
Saved VAE -> /content/drive/MyDrive/image_compression_project/models/vae_epoch10.pth


In [8]:

from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import os
import torch
import numpy as np
from skimage.metrics import peak_signal_noise_ratio as psnr, structural_similarity as ssim
import torchvision.utils as vutils

transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor()
])

test_ds = datasets.STL10(root='data', split='test', download=True, transform=transform)
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False)
print("Test dataset loaded:", len(test_ds), "images")

os.makedirs(os.path.join(BASE, "results/grids"), exist_ok=True)

vae.eval()
psnrs, ssims = [], []

for i, (img, _) in enumerate(test_loader):
    if i >= 24:
        break

    img = img.to(device)
    with torch.no_grad():
        rec, _, _ = vae(img)
    orig_np = img.cpu().squeeze(0).permute(1, 2, 0).numpy()
    rec_np  = rec.cpu().squeeze(0).permute(1, 2, 0).numpy()

    # Compute PSNR & SSIM
    psnr_val = psnr(orig_np, rec_np, data_range=1.0)
    ssim_val = ssim(orig_np, rec_np, channel_axis=-1, data_range=1.0, win_size=7)

    psnrs.append(psnr_val)
    ssims.append(ssim_val)

    # Save side-by-side image (Original | Reconstruction)
    grid_path = os.path.join(BASE, f"results/grids/vae_grid_{i+1:02d}.png")
    vutils.save_image(torch.cat([img.cpu(), rec.cpu()], dim=3), grid_path)


print(f"\n VAE Avg PSNR: {np.mean(psnrs):.2f}")
print(f"VAE Avg SSIM: {np.mean(ssims):.3f}")

# Save first sample separately for quick viewing ---
sample_path = os.path.join(BASE, "results/grids/vae_sample01.png")
!cp {sample_path} {sample_path}
print(f" Sample saved for demo: {sample_path}")


100%|██████████| 2.64G/2.64G [00:36<00:00, 72.7MB/s]


Test dataset loaded: 8000 images

 VAE Avg PSNR: 18.46
VAE Avg SSIM: 0.398
cp: cannot stat '/content/drive/MyDrive/image_compression_project/results/grids/vae_sample01.png': No such file or directory
 Sample saved for demo: /content/drive/MyDrive/image_compression_project/results/grids/vae_sample01.png


In [9]:
torch.save(vae.state_dict(), "/content/drive/MyDrive/image_compression_project/models/vae_epoch10.pth")

In [10]:
!ls /content/drive/MyDrive/image_compression_project/models

ae_epoch8.pth  vae_epoch10.pth
